## **Bank Nifty Predictability Under Regulatory Expiry-Structure Reform**

### **Notebook 08: Hypothesis Testing and Evaluation**

[![Stats](https://img.shields.io/badge/Statistical--Test-Diebold--Mariano-blue)](https://www.scipy.org/)

---

**Scope:** Run the Diebold-Mariano statistical significance test to compare forecast errors of the baseline and proposed models.

---


### **Pipeline Context**

This is a sequential 9-notebook experimental workflow:

1. **01_data_collection.ipynb**
2. **02_data_cleaning_eda.ipynb**
3. **03_feature_engineering.ipynb**
4. **04_volatility_regime_detection.ipynb**
5. **05_baseline_models.ipynb**
6. **06_proposed_model.ipynb**
7. **07_ablation_study.ipynb**
8. **08_evaluation_and_diagnostics.ipynb**
9. **09_figures_for_paper.ipynb**

---
**Current Location:** 8. **08_evaluation_and_diagnostics.ipynb** <- *You are here*

---


## 1. Introduction

### 1.1 Purpose of This Notebook
To prove that the proposed model's forecasting improvement over the baseline is statistically meaningful (not just a numerical artifact), we implement and run a formal **Diebold-Mariano test** comparing the loss differentials of both models.

### 1.2 Hypothesis Framework
- **Null Hypothesis (H₀):** The baseline and proposed models have equal predictive accuracy (mean loss differential is zero).
- **Alternative Hypothesis (H₁):** The proposed model has superior predictive accuracy (mean loss differential is positive, indicating lower errors).

## 2. Environment Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd
import joblib
import json

project_dir = Path("e:/BN_Structure_Reforms/banknifty-expiry-volatility-ml")
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

from src import config
from src.evaluation import diebold_mariano_test

## 3. Load Predictions

In [2]:
test = pd.read_parquet(config.PROCESSED_DATA_DIR / "test.parquet")
y_test = test['Target'].values

predictions = joblib.load(config.MODELS_DIR / "ablation_predictions.joblib")
baseline_preds = predictions["Baseline (Technical Only)"]
proposed_preds = predictions["Proposed (Combined)"]

## 4. Run Diebold-Mariano Tests

In [3]:
dm_mse = diebold_mariano_test(y_test, baseline_preds, proposed_preds, power=2)
dm_mae = diebold_mariano_test(y_test, baseline_preds, proposed_preds, power=1)

print("DM Test (MSE Loss):", dm_mse)
print("DM Test (MAE Loss):", dm_mae)

DM Test (MSE Loss): {'DM_Stat': -0.40349434219614677, 'P_Value_Two_Sided': 0.6865845960864614, 'P_Value_One_Sided': 0.6567077019567693}
DM Test (MAE Loss): {'DM_Stat': 0.22544877293416235, 'P_Value_Two_Sided': 0.8216301727759254, 'P_Value_One_Sided': 0.4108150863879627}


## 5. Export Test Metrics

In [4]:
# Import backtesting function
from src.evaluation import backtest_trading_strategy

# Backtest all conditions
backtest_results = {}
for name, preds in predictions.items():
    res = backtest_trading_strategy(y_test, preds, cost_rate=0.0002)
    # Remove large Net_Returns list to keep JSON file compact
    net_ret = res.pop("Net_Returns")
    backtest_results[name] = res

# Console output
print("--- Trading Strategy Backtest (Net of 0.02% transaction costs) ---")
for name, res in backtest_results.items():
    print(f"{name:<30} | Sharpe: {res['Sharpe_Ratio']:.4f} | Sortino: {res['Sortino_Ratio']:.4f} | Max DD: {res['Max_Drawdown']:.4%}")

summary = {
    "Diebold_Mariano_MSE": dm_mse,
    "Diebold_Mariano_MAE": dm_mae,
    "Trading_Backtest": backtest_results
}
with open(config.RESULTS_DIR / "metrics_summary.json", "w") as f:
    json.dump(summary, f, indent=4)
print("[OK] Saved metrics summary to JSON.")

--- Trading Strategy Backtest (Net of 0.02% transaction costs) ---
Baseline (Technical Only)      | Sharpe: 1.9359 | Sortino: 2.9449 | Max DD: -7.6374%
Baseline + Expiry              | Sharpe: 2.1010 | Sortino: 3.0811 | Max DD: -6.6316%
Baseline + Regime              | Sharpe: 1.4303 | Sortino: 2.1249 | Max DD: -10.4969%
Proposed (Combined)            | Sharpe: 2.0767 | Sortino: 3.0689 | Max DD: -6.6316%
[OK] Saved metrics summary to JSON.
